# Classifying Text

source: https://github.com/keithgalli/sklearn

In [23]:
class Sentiment:
    NEGATIVE = "NEGATIVE"
    NEUTRAL = "NEUTRAL"
    POSITIVE = "POSITIVE"

In [24]:
class Review:
    def __init__(self, text, score):
        self.text = text
        self.score = score
        self.sentiment = self.get_sentiment()
        
    def get_sentiment(self):
        if self.score <= 2:
            return Sentiment.NEGATIVE
        elif self.score == 3:
            return Sentiment.NEUTRAL
        else:
            return Sentiment.POSITIVE

In [99]:
import random

class ReviewContainer:
    def __init__(self, reviews):
        self.reviews = reviews
        
    def get_text(self) -> list:
        return [x.text for x in self.reviews]
    
    def get_sentiment(self) -> list:
        return [x.sentiment for x in self.reviews]
        
    def evenly_distribute(self):
        negative = list(filter(lambda x: x.sentiment == Sentiment.NEGATIVE, self.reviews))
        neutral = list(filter(lambda x: x.sentiment == Sentiment.NEUTRAL, self.reviews))
        positive = list(filter(lambda x: x.sentiment == Sentiment.POSITIVE, self.reviews))
        
        min_len = min(len(negative), len(neutral), len(positive))
        
        negative_shrunk = negative[:min_len]
        neutral_shrunk = neutral[:min_len]
        positive_shrunk = positive[:min_len]
        
        self.reviews = negative_shrunk + neutral_shrunk + positive_shrunk
        random.shuffle(self.reviews)

In [62]:
import json

file_name = "./data/Books_small_10000.json"

reviews = []
with open(file_name) as f:
    for line in f:
        review = json.loads(line)
        reviews.append(Review(review["reviewText"], review["overall"]))

In [63]:
reviews[5].text

'I hoped for Mia to have some peace in this book, but her story is so real and raw.  Broken World was so touching and emotional because you go from Mia\'s trauma to her trying to cope.  I love the way the story displays how there is no "just bouncing back" from being sexually assaulted.  Mia showed us how those demons come for you every day and how sometimes they best you. I was so in the moment with Broken World and hurt with Mia because she was surrounded by people but so alone and I understood her feelings.  I found myself wishing I could give her some of my courage and strength or even just to be there for her.  Thank you Lizzy for putting a great character\'s voice on a strong subject and making it so that other peoples story may be heard through Mia\'s.'

In [64]:
from sklearn.model_selection import train_test_split

reviews_train, reviews_test = train_test_split(
    reviews, test_size=0.2, random_state=42
)

In [65]:
len(reviews_train), len(reviews_test)

(8000, 2000)

In [66]:
reviews_train[0].sentiment

'POSITIVE'

In [100]:
container_train = ReviewContainer(reviews_train)
container_train.evenly_distribute()
len(container_train.reviews)

1539

In [101]:
container_test = ReviewContainer(reviews_test)
container_test.evenly_distribute()
len(container_test.reviews)

393

In [102]:
train_x = container_train.get_text()
train_y = container_train.get_sentiment()

test_x = container_test.get_text()
test_y = container_test.get_sentiment()

In [103]:
train_y.count(Sentiment.POSITIVE), train_y.count(Sentiment.NEUTRAL), train_y.count(Sentiment.NEGATIVE)

(513, 513, 513)

In [119]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

corpus = train_x
#vectorizer = CountVectorizer()

vectorizer = TfidfVectorizer()

train_x_v = vectorizer.fit_transform(corpus)
test_x_v = vectorizer.transform(test_x)
train_x_v

<1539x12343 sparse matrix of type '<class 'numpy.float64'>'
	with 98504 stored elements in Compressed Sparse Row format>

In [120]:
print(train_x[0])
print(train_x_v[0].toarray())

For the most part, the story was good. Not over the top great but it kept me reading to see how it all played out. I really liked Symon and his patience and strength when interacting with Elena. As for Elena, she was torn between doing what was right and wanting to be as normal as she could. She could have trusted Symon way sooner because it got old having to read about her second guessing him when it was clear he was one who could be trusted.What I didn't care for was the author's frequent explanations and insertions into the story about the characters feelings. It felt forced. The interactions between the characters was enough to keep the story moving without all the extra.This was in a set of Five Unforgettable Knights and I had never read this author before. I don't buy into the healing processes Elena used and would typically not read books like this, but since the description didn't mention anything about it, I didn't consider that when I gave it three stars.
[[0. 0. 0. ... 0. 0.

In [121]:
from sklearn.svm import SVC

svc_model = SVC(kernel='linear')
svc_model.fit(train_x_v, train_y)

SVC(kernel='linear')

In [122]:
svc_pred = svc_model.predict(test_x_v)

In [123]:
from sklearn.metrics import f1_score, classification_report

print(classification_report(test_y, svc_pred))

              precision    recall  f1-score   support

    NEGATIVE       0.59      0.60      0.60       131
     NEUTRAL       0.52      0.50      0.51       131
    POSITIVE       0.68      0.69      0.69       131

    accuracy                           0.60       393
   macro avg       0.60      0.60      0.60       393
weighted avg       0.60      0.60      0.60       393



In [124]:
f1_score(svc_pred, test_y, average=None, 
         labels=[Sentiment.POSITIVE, Sentiment.NEUTRAL, Sentiment.NEGATIVE]
)

array([0.68939394, 0.50583658, 0.59622642])

In [125]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(train_x_v.toarray(), train_y)

GaussianNB()

In [126]:
nb_pred = nb_model.predict(test_x_v.toarray())
f1_score(nb_pred, test_y, average=None, 
         labels=[Sentiment.POSITIVE, Sentiment.NEUTRAL, Sentiment.NEGATIVE]
)

array([0.5631769 , 0.4063745 , 0.42635659])

In [127]:
prompts = ["Wasn't that bad, but I enjoyed this"]
prompts = vectorizer.transform(prompts)
svc_model.predict(prompts)

array(['NEUTRAL'], dtype='<U8')

## grid_search

In [132]:
from sklearn.model_selection import GridSearchCV

grid_model = GridSearchCV(
    estimator=SVC(),
    param_grid={
        'kernel': ['linear', 'rbf', 'sigmoid'],
        'C': [1, 2, 4, 8, 12, 32]
    },
    cv=3,
)
grid_model.fit(train_x_v, train_y)

GridSearchCV(cv=3, estimator=SVC(),
             param_grid={'C': [1, 2, 4, 8, 12, 32],
                         'kernel': ['linear', 'rbf', 'sigmoid']})

In [134]:
svc_model_best = grid_model.best_estimator_

In [137]:
svc_pred = svc_model.predict(test_x_v)
print(classification_report(test_y, svc_pred))

              precision    recall  f1-score   support

    NEGATIVE       0.59      0.60      0.60       131
     NEUTRAL       0.52      0.50      0.51       131
    POSITIVE       0.68      0.69      0.69       131

    accuracy                           0.60       393
   macro avg       0.60      0.60      0.60       393
weighted avg       0.60      0.60      0.60       393



In [136]:
svc_pred = svc_model_best.predict(test_x_v)
print(classification_report(test_y, svc_pred))

              precision    recall  f1-score   support

    NEGATIVE       0.59      0.60      0.60       131
     NEUTRAL       0.53      0.47      0.50       131
    POSITIVE       0.70      0.76      0.73       131

    accuracy                           0.61       393
   macro avg       0.61      0.61      0.61       393
weighted avg       0.61      0.61      0.61       393



## saving model

In [139]:
import pickle

with open('./models/sentiment_classifier.plk', 'wb') as f:
    pickle.dump(svc_model_best, f)

In [140]:
with open('./models/sentiment_classifier.plk', 'rb') as f:
    svc_model_loaded = pickle.load(f)

In [141]:
svc_pred = svc_model_loaded.predict(test_x_v)
print(classification_report(test_y, svc_pred))

              precision    recall  f1-score   support

    NEGATIVE       0.59      0.60      0.60       131
     NEUTRAL       0.53      0.47      0.50       131
    POSITIVE       0.70      0.76      0.73       131

    accuracy                           0.61       393
   macro avg       0.61      0.61      0.61       393
weighted avg       0.61      0.61      0.61       393

